# Fine-tune EOMT segmentation model with DINOv3
---
We can run this notebook to go through fine-tuning a pre-trained EOMT segmentation model on our custom tactile paving dataset using the latest DINOv3 weights as the backbone feature extractor.

Using the trained model, we can then run inference on our test dataset of real tactile paving data to get a picture of how the model's segmentations would work in real life scenarios.

## Data

Please request permission access from the forms provided to download both our custom synthetically augmented dataset as well as our collected real-world testset that will be used for training and evalutation respectively.

In order to train the model, the datasets must be arranged in a specific format. For EOMT this is the recommended structure that will be used:
```
my_data_dir
├── train
│   ├── images
│   │   ├── image0.jpg
│   │   └── image1.jpg
│   └── masks
│       ├── image0.png
│       └── image1.png
└── val
    ├── images
    |  ├── image2.jpg
    |  └── image3.jpg
    └── masks
       ├── image2.png
       └── image3.png
```

where each jpg image has a corresponding png mask with the same filename. 

## Setup

To use DINOV3, please sign up and accept the terms of use from Meta to get access to the DINOv3 checkpoints and their download links. 

We can set our DINOV3 model to use here. For the purpose of this, we are goning to use the 'DINOV3-S' i.e. small model.


In [ ]:
dinov3_model_url = "..." # add the download link here

To train the EOMT model with the DIONV3 weights, we are going to use the Lightly-Train library which has built in support for semantic segmentation training. Be sure to install and import the latest version for DINOV3 compatibility.

In [ ]:
pip install lightly-train==0.11.2

In [ ]:
import lightly_train

## Training

Now that we have the data and lightly-train setup correctly, we can begin training:

In [ ]:
if __name__ == "__main__":
    lightly_train.train_semantic_segmentation(
        out="out/...", # Path to output directly where checkpoints and logs will be stored
        model="dinov3/vits16-eomt",
        model_args={
            # Replace with your own url
            "backbone_url": dinov3_model_url,
        },
        data={
            "train": {
                "images": "...",   # Path to training images
                "masks": "...",     # Path to training masks
            },
            "val": {
                "images": "...",     # Path to validation images
                "masks": "...",       # Path to validation masks
            },
            "classes": {                       # Classes in the dataset                    
                0: "background",
                255: "tactile-paving",
                # ...
            },
            # Optional 
            # "ignore_classes": [0], # Classes that are in the dataset but should be ignored during training.
            # 
        },
        resume_interrupted=True,
        transform_args={
            "image_size": (518, 518), # (height, width)
            "normalize": {
                "mean": [0.485, 0.456, 0.406],
                "std": [0.229, 0.224, 0.225],
            },
        },
    )

**Note:**
- It is recommended to train the model on segementing the background class as well for best results. In our dataset the masks have the tactile-paving objects at pixel value 255 i.e. binary 1, and the backgrounds at 0.
- Based on resources available, it is recommended to use multiple GPUs to speed up the training process. 
- Logging can configured with the 'logger_args' argument. All tensorboard logs are available in the specified out directory.
- Extensive Documentation for the training parameters can be accessed here: [lightly_train_doc](https://docs.lightly.ai/train/stable/python_api/lightly_train.html) 

## Inference

Once training is done, we can run inference with the model on our test set.

First, load the model:

In [ ]:
model = lightly_train.load_model_from_checkpoint(
    "..."
) # path to checkpoint file

We can now use a sample image from our test set and produce a segmented mask on it showing the detected tactile paving. 

We can evaluate the accuracy of it by comparing it with the ground truth mask of the test image. 

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.utils import draw_segmentation_masks
from PIL import Image
import numpy as np

test_image_fpath = "..." # path of test image
test_mask_fpath = '...' # path of test image gt mask

test_image = Image.open(test_image_fpath)
gt_mask = Image.open(test_mask_fpath)

masks = model.predict(test_image)
masks = torch.stack([masks == class_id for class_id in masks.unique()]).detach().cpu().numpy()
pred_mask = Image.fromarray(masks[-1]) # Gettig the tactile paving segmentations, not the background segemntations

plt.figure(figsize=(6, 2), dpi=300)
plt.subplot(1, 3, 1)
plt.axis('off')
plt.imshow(test_image)
plt.title('input image')

plt.subplot(1, 3, 2)
plt.axis('off')
plt.imshow(gt_mask, cmap = 'gray')
plt.title('ground_truth')

plt.subplot(1, 3, 3)
plt.axis('off')
plt.imshow(pred_mask)
plt.title('prediction')

plt.show()


